In [1]:
import random
import numpy as np

# Tham số bài toán
TEACHERS = ['Thầy An', 'Cô Bình', 'Thầy Cường', 'Cô Diệp', 'Thầy Đông']
SUBJECTS = ['Toán', 'Tin học']
CLASSES = ['10A1', '10A2', '11B1', '11B2', '12C1']
DAYS = ['Thứ 2', 'Thứ 3', 'Thứ 4', 'Thứ 5', 'Thứ 6', 'Thứ 7']
SLOTS_PER_DAY = 4  # Giả sử 4 tiết mỗi buổi

# Mỗi lớp cần 4 tiết Toán và 2 tiết Tin mỗi tuần
REQUIREMENTS = {'Toán': 4, 'Tin học': 2}

print(f'Số giáo viên: {len(TEACHERS)}')
print(f'Số lớp: {len(CLASSES)}')

Số giáo viên: 5
Số lớp: 5


In [2]:
class ScheduleGA:
    def __init__(self, pop_size=50, mutation_rate=0.1):
        self.pop_size = pop_size
        self.mutation_rate = mutation_rate

    def create_individual(self):
        # Một cá thể là danh sách các gene: (Lớp, Môn, Giáo viên, Thứ, Tiết)
        individual = []
        for cls in CLASSES:
            for sub, count in REQUIREMENTS.items():
                for _ in range(count):
                    gene = {
                        'class': cls,
                        'subject': sub,
                        'teacher': random.choice(TEACHERS),
                        'day': random.choice(DAYS),
                        'slot': random.randint(1, SLOTS_PER_DAY)
                    }
                    individual.append(gene)
        return individual

    def fitness(self, individual):
        conflicts = 0
        # Ràng buộc 1: Một giáo viên không thể dạy 2 lớp cùng lúc
        # Ràng buộc 2: Một lớp không thể học 2 môn cùng lúc
        time_slots_teacher = {}
        time_slots_class = {}

        for gene in individual:
            t_key = (gene['teacher'], gene['day'], gene['slot'])
            c_key = (gene['class'], gene['day'], gene['slot'])

            if t_key in time_slots_teacher: conflicts += 1
            else: time_slots_teacher[t_key] = True

            if c_key in time_slots_class: conflicts += 1
            else: time_slots_class[c_key] = True

        return 1 / (1 + conflicts)

    def evolve(self, generations=100):
        population = [self.create_individual() for _ in range(self.pop_size)]

        for gen in range(generations):
            population = sorted(population, key=lambda x: self.fitness(x), reverse=True)
            if self.fitness(population[0]) == 1.0: break

            next_gen = population[:10] # Elitism
            while len(next_gen) < self.pop_size:
                parent1, parent2 = random.sample(population[:20], 2)
                child = self.crossover(parent1, parent2)
                self.mutate(child)
                next_gen.append(child)
            population = next_gen

        return population[0]

    def crossover(self, p1, p2):
        point = random.randint(0, len(p1)-1)
        return p1[:point] + p2[point:]

    def mutate(self, individual):
        if random.random() < self.mutation_rate:
            idx = random.randint(0, len(individual)-1)
            individual[idx]['day'] = random.choice(DAYS)
            individual[idx]['slot'] = random.randint(1, SLOTS_PER_DAY)

ga = ScheduleGA()
best_schedule = ga.evolve(generations=200)
print(f'Độ thích nghi tốt nhất: {ga.fitness(best_schedule)}')


Độ thích nghi tốt nhất: 1.0


In [3]:
import pandas as pd
df_result = pd.DataFrame(best_schedule)
display(df_result.sort_values(by=['day', 'slot']).head(10))

,class,subject,teacher,day,slot
4,10A1,Tin học,Cô Diệp,Thứ 2,2
12,11B1,Toán,Cô Diệp,Thứ 2,3
19,11B2,Toán,Thầy An,Thứ 2,3
5,10A1,Tin học,Thầy Cường,Thứ 2,4
10,10A2,Tin học,Thầy Cường,Thứ 3,1
14,11B1,Toán,Cô Diệp,Thứ 3,1
11,10A2,Tin học,Thầy An,Thứ 3,3
15,11B1,Toán,Cô Bình,Thứ 3,3
2,10A1,Toán,Thầy Đông,Thứ 3,4
24,12C1,Toán,Cô Diệp,Thứ 3,4
